# III. Comparaison des tryptiques.csv

## 1) Imports et chemins

In [95]:
import pandas as pd
import numpy as np

TRIPTYQUES_FOLDER = "../results/csv_triptyques/"

auto = pd.read_csv(f"{TRIPTYQUES_FOLDER}triptyques_tdm_chap.csv")
manuel = pd.read_csv(F"{TRIPTYQUES_FOLDER}triptyques_chap1to5_sacr.csv")

## 2) Est ce que les deux fichier on les memes colonnes ?

In [96]:

print("Colonnes automatique :")
print(auto.columns.tolist())

print("\nColonnes manuel :")
print(manuel.columns.tolist())


Colonnes automatique :
['Phrase', 'Sujet', 'Verbe', 'Objet', 'Dep_sujet', 'Dep_verbe', 'Dep_objet', 'ID_sujet', 'ID_verbe', 'ID_objet', 'IDs_sujet', 'IDs_objet', 'Lemme_verbe', 'Head_verbe', 'Annotation_verbe_col8', 'Annotation_verbe_col9', 'Num_phrase', 'Num_paragr', 'Sentence_index', 'personnages_sujet', 'personnages_objet', 'personnages_triptyque', 'lieux_sujet', 'lieux_objet', 'lieux_triptyque', 'types_lieux_triptyque', 'a_personnage_triptyque', 'a_lieu_triptyque', 'a_personnage_et_lieu_triptyque', 'chapter']

Colonnes manuel :
['Phrase', 'Sujet', 'Verbe', 'Objet', 'Dep_sujet', 'Dep_verbe', 'Dep_objet', 'ID_sujet', 'ID_verbe', 'ID_objet', 'IDs_sujet', 'IDs_objet', 'Lemme_verbe', 'Head_verbe', 'Annotation_verbe_col8', 'Annotation_verbe_col9', 'Num_phrase', 'Num_paragr', 'Sentence_index', 'personnages_sujet', 'personnages_objet', 'personnages_triptyque', 'lieux_sujet', 'lieux_objet', 'lieux_triptyque', 'types_lieux_triptyque', 'a_personnage_triptyque', 'a_lieu_triptyque', 'a_personna

## 3) Filter csv automatiques par rapport aux chapitres annotés manuellement (1 à 5 pour l'instant) 

In [97]:
auto_chap1to5 = auto[auto["chapter"].isin([1, 2, 3, 4, 5])]

print("Triptyques automatiques avant filtre :", len(auto))
print("Triptyques automatiques après filtre chap. 1 à 5 :", len(auto_chap1to5))
print("Triptyques manuels chap. 1 à 5 :", len(manuel))

Triptyques automatiques avant filtre : 13047
Triptyques automatiques après filtre chap. 1 à 5 : 1259
Triptyques manuels chap. 1 à 5 : 1255


## 4) Création d'une clé pour chaque tryptique

In [98]:
key_cols = ["Phrase", "ID_sujet", "ID_verbe", "ID_objet"]

for df in [auto_chap1to5, manuel]:
    df["cle_triptyque"] = (
        df[key_cols]
        .fillna("")
        .astype(str)
        .apply(lambda row: " || ".join(row), axis=1)
    )

## 5) Comparaison des clés

In [99]:
cles_auto = set(auto_chap1to5["cle_triptyque"])
cles_manuel = set(manuel["cle_triptyque"])

communs = cles_auto & cles_manuel
seulement_auto = cles_auto - cles_manuel
seulement_manuel = cles_manuel - cles_auto

print("Triptyques communs :", len(communs))
print("Triptyques seulement automatiques :", len(seulement_auto))
print("Triptyques seulement manuels :", len(seulement_manuel))

Triptyques communs : 204
Triptyques seulement automatiques : 1055
Triptyques seulement manuels : 1051


## 6) Quelles sont les dépéndances ou il y a le plus de différences ?

In [100]:
# On récupère les triptyques produits uniquement par l'annotation automatique.
df_seulement_auto = auto_chap1to5[auto_chap1to5["cle_triptyque"].isin(seulement_auto)].copy()
df_seulement_auto["source_comparaison"] = "seulement_automatique"

# On récupère les triptyques présents uniquement dans l'annotation manuelle.
df_seulement_manuel = manuel[manuel["cle_triptyque"].isin(seulement_manuel)].copy()
df_seulement_manuel["source_comparaison"] = "seulement_manuel"

# On rassemble les deux types de différences dans un seul tableau.
differences_triptyques = pd.concat(
    [df_seulement_auto, df_seulement_manuel],
    ignore_index=True
)

# On compte les différences par combinaison de dépendances :
# dépendance du sujet, dépendance du verbe, dépendance de l'objet.
resume_dependances = (
    differences_triptyques
    .groupby(["source_comparaison", "Dep_sujet", "Dep_verbe", "Dep_objet"])
    .size()
    .reset_index(name="nombre")
    .sort_values("nombre", ascending=False)
)

resume_dependances

,source_comparaison,Dep_sujet,Dep_verbe,Dep_objet,nombre
98,seulement_manuel,nsubj,root,obl:mod,82
37,seulement_automatique,nsubj,root,obl:mod,79
35,seulement_automatique,nsubj,root,obj,76
96,seulement_manuel,nsubj,root,obj,73
8,seulement_automatique,nsubj,acl:relcl,obj,47
...,...,...,...,...,...
114,seulement_manuel,nsubj:pass,root,obl:agent,1
113,seulement_manuel,nsubj:pass,root,iobj,1
118,seulement_manuel,nsubj:pass,xcomp,obl:mod,1
119,seulement_manuel,obl:arg,acl:relcl,iobj,1


In [101]:
df_seulement_auto

,Phrase,Sujet,Verbe,Objet,Dep_sujet,Dep_verbe,Dep_objet,ID_sujet,ID_verbe,ID_objet,...,lieux_sujet,lieux_objet,lieux_triptyque,types_lieux_triptyque,a_personnage_triptyque,a_lieu_triptyque,a_personnage_et_lieu_triptyque,chapter,cle_triptyque,source_comparaison
0,"En l’ année 1872, la maison portant le numéro ...",NaN,portant,le numéro 7 de Saville-row Burlington Gardens–...,NaN,acl,obj,NaN,8,10.0,...,[],"['saville - row', 'burlington gardens']","['saville - row', 'burlington gardens']",['FAC'],False,True,False,1,"En l’ année 1872, la maison portant le numéro ...",seulement_automatique
1,"En l’ année 1872, la maison portant le numéro ...",Sheridan,mourut,en 1814,nsubj,acl:relcl,obl:mod,23.0,24,26.0,...,[],[],[],[],False,False,False,1,"En l’ année 1872, la maison portant le numéro ...",seulement_automatique
2,"En l’ année 1872, la maison portant le numéro ...",la maison maison,habitée,En l’année 1872,nsubj:pass,root,obl:mod,7.0,30,3.0,...,['la maison'],[],['la maison'],['FAC'],False,True,False,1,"En l’ année 1872, la maison portant le numéro ...",seulement_automatique
3,"En l’ année 1872, la maison portant le numéro ...",la maison maison,habitée,par Phileas Fogg esq.,nsubj:pass,root,obl:agent,7.0,30,32.0,...,['la maison'],[],['la maison'],['FAC'],True,True,True,1,"En l’ année 1872, la maison portant le numéro ...",seulement_automatique
4,"En l’ année 1872, la maison portant le numéro ...",la maison maison,habitée,l’un des membres les plus singuliers et les pl...,nsubj:pass,root,obl:mod,7.0,30,39.0,...,['la maison'],[],['la maison'],['FAC'],False,True,False,1,"En l’ année 1872, la maison portant le numéro ...",seulement_automatique
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1254,On rappela ce que l’ existence de Phileas Fogg...,NaN,prétextant,un voyage,NaN,advmod,obj,NaN,29,31.0,...,[],[],[],[],False,False,False,5,On rappela ce que l’ existence de Phileas Fogg...,seulement_automatique
1255,On rappela ce que l’ existence de Phileas Fogg...,NaN,appuyant,l’,NaN,conj,obj,NaN,37,36.0,...,[],[],[],[],False,False,False,5,On rappela ce que l’ existence de Phileas Fogg...,seulement_automatique
1256,On rappela ce que l’ existence de Phileas Fogg...,NaN,appuyant,sur un pari insensé,NaN,conj,obl:arg,NaN,37,40.0,...,[],[],[],[],False,False,False,5,On rappela ce que l’ existence de Phileas Fogg...,seulement_automatique
1257,On rappela ce que l’ existence de Phileas Fogg...,ce personnage,eu,d’autre but,nsubj,ccomp,obj,27.0,45,48.0,...,[],[],[],[],True,False,False,5,On rappela ce que l’ existence de Phileas Fogg...,seulement_automatique
